# Agent demo: agents ↔ MCP server ↔ feature store and drift detection

Three agents run on a local model (Qwen2.5-3B on llama.cpp). They never read data directly: every number comes from a tool on the MCP server.

| Tool | Behind it |
|---|---|
| `get_pair_features` | `signal_api` → Feast online store (Redis) → model |
| `check_regime` | `regime_api` → drift detection on warehouse prices |

Needs running: `signal_api` (8000), `regime_api` (8001), the MCP server (8002) and llama.cpp (8085). See `docs/llm_agents.md`.

In [1]:
import sys

sys.path.insert(0, "..")

import time
import warnings

import httpx
from fastmcp import Client

from services.agent.analysts import build_agents
from services.agent.llm import OpenAICompatClient
from services.agent.mcp_tools import load_mcp_tools
from services.agent.telemetry import AgentTelemetry

warnings.filterwarnings("ignore")
MCP_URL = "http://localhost:8002/mcp"
LLM_URL = "http://localhost:8085"

## 1. Connect to the MCP server and list its tools

In [2]:
mcp = Client(MCP_URL)
await mcp.__aenter__()
tools = {tool.name: tool for tool in await load_mcp_tools(mcp)}
for name, tool in sorted(tools.items()):
    print(f"{name}: {tool.description.splitlines()[0]}")

check_regime: Drift check for a pair: regime (stable, shifting, broken), coint_pvalue and psi.
get_pair_features: Latest features for a pair and the model's take/skip decision.


## 2. Call the tools directly over MCP

This is what an agent receives. `get_pair_features` returns features pulled from the Feast online store plus the model's decision; `check_regime` returns the drift check.

In [3]:
await tools["get_pair_features"].call(pair_id="KO__PEP")

{'pair_id': 'KO__PEP',
 'prob': 0.688019266240321,
 'take_trade': True,
 'threshold': 0.5,
 'model_version': 'meta_label-v4',
 'features': {'zscore': 0.7221475075235323,
  'hedge_ratio': 0.33932791182905125,
  'spread_vol': 11.158223471472706,
  'correlation_60d': 0.6930732367342185}}

In [4]:
await tools["check_regime"].call(pair_id="KO__PEP")

{'pair_id': 'KO__PEP',
 'regime': 'stable',
 'coint_pvalue': 0.009517318159137825,
 'psi': 0.12288401017146858,
 'bars_used': 315}

## 3. Build the agents

Temperature 0 and a fixed seed, so the same question gives the same answer.

In [5]:
http = httpx.AsyncClient()
telemetry = AgentTelemetry()
agents = build_agents(
    OpenAICompatClient(http, base_url=LLM_URL),
    tools["get_pair_features"],
    tools["check_regime"],
    telemetry,
)


async def ask(agent: str, question: str) -> None:
    started = time.perf_counter()
    answer = await agents[agent].run(question)
    print(f"[{agent}] {time.perf_counter() - started:.1f}s, {answer.steps} steps, "
          f"{answer.input_tokens}+{answer.output_tokens} tokens")
    for call in answer.tool_calls:
        status = "FAILED" if call.failed else "ok"
        print(f"  tool {call.name}({call.arguments}) -> {status}")
    print("  answer:", answer.text)


sorted(agents)

['coordinator', 'features_analyst', 'regime_analyst']

## 4. Features analyst: pulls features from the feature store through MCP

In [6]:
await ask("features_analyst", "Report on pair GS__MS.")

[features_analyst] 8.6s, 2 steps, 789+106 tokens
  tool get_pair_features({'pair_id': 'GS__MS'}) -> ok
  answer: The z-score for the pair GS__MS is 2.75. The model's probability is 0.585. Given the model's threshold of 0.5, the model would take the trade. The hedge ratio is 0.409, spread volatility is 7.86, and the 60-day correlation is 0.542.


## 5. Regime analyst: drift detection through MCP

In [7]:
await ask("regime_analyst", "What is the regime of KO__PEP?")

[regime_analyst] 3.6s, 2 steps, 696+59 tokens
  tool check_regime({'pair_id': 'KO__PEP'}) -> ok
  answer: The regime of KO__PEP is stable with a cointegration p-value of 0.0095 and PSI of 0.12.


## 6. Coordinator: delegates to both analysts and combines their reports

In [8]:
await ask("coordinator", "Is KO__PEP still tradeable?")

[coordinator] 15.9s, 2 steps, 904+119 tokens
  tool ask_features_analyst({'pair_id': 'KO__PEP'}) -> ok
  tool ask_regime_analyst({'pair_id': 'KO__PEP'}) -> ok
  answer: The pair KO__PEP is still tradeable as the features analyst indicates a model probability of 0.688, suggesting a favorable trade. The regime analyst confirms the regime is stable with a cointegration p-value of 0.0095 and PSI of 0.12.


## 7. When things go wrong

An unknown pair: the tool returns an error and the agent says so instead of inventing numbers.

In [9]:
await ask("regime_analyst", "What is the regime of NOPE__XYZ?")

[regime_analyst] 2.5s, 2 steps, 638+48 tokens
  tool check_regime({'pair_id': 'NOPE__XYZ'}) -> FAILED
  answer: There are no prices available for the pair NOPE__XYZ. Please check the pair ID and try again.


A prompt containing personal data is refused before it reaches the model.

In [10]:
await ask("coordinator", "Email hoa@example.com whether KO__PEP is tradeable.")

[coordinator] 0.0s, 0 steps, 0+0 tokens
  answer: Request refused: it contains personal data (email).


## 8. Telemetry collected during this notebook

In [11]:
from prometheus_client import generate_latest

wanted = ("agent_calls_total{", "agent_tool_calls_total{", "agent_tool_failures_total{",
          "llm_tokens_total{", "llm_round_trip_seconds_count", "llm_round_trip_seconds_sum",
          "llm_time_to_first_token_seconds_count", "llm_time_to_first_token_seconds_sum",
          "agent_pii_blocked_total{")
for line in generate_latest(telemetry.registry).decode().splitlines():
    if line.startswith(wanted):
        print(line)

agent_calls_total{agent="features_analyst"} 2.0
agent_calls_total{agent="regime_analyst"} 3.0
agent_calls_total{agent="coordinator"} 2.0
agent_tool_calls_total{tool="get_pair_features"} 2.0
agent_tool_calls_total{tool="check_regime"} 3.0
agent_tool_calls_total{tool="ask_features_analyst"} 1.0
agent_tool_calls_total{tool="ask_regime_analyst"} 1.0
agent_tool_failures_total{tool="check_regime"} 1.0
llm_tokens_total{kind="input",model="/models/qwen2.5-3b-instruct-q4_k_m.gguf"} 4514.0
llm_tokens_total{kind="output",model="/models/qwen2.5-3b-instruct-q4_k_m.gguf"} 455.0
llm_round_trip_seconds_count{model="/models/qwen2.5-3b-instruct-q4_k_m.gguf"} 12.0
llm_round_trip_seconds_sum{model="/models/qwen2.5-3b-instruct-q4_k_m.gguf"} 30.371578252874315
llm_time_to_first_token_seconds_count{model="/models/qwen2.5-3b-instruct-q4_k_m.gguf"} 12.0
llm_time_to_first_token_seconds_sum{model="/models/qwen2.5-3b-instruct-q4_k_m.gguf"} 12.461297
agent_pii_blocked_total{kind="email"} 1.0


In [12]:
await mcp.__aexit__(None, None, None)
await http.aclose()